# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdubakr77/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df.columns.tolist())
print(df.shape)
df.head(3)

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split design:** grouped split by `client_id`, not a random row split, so no client's pages appear in both train and test. This matches the leakage-prevention approach from the starter pipeline (Notebook 3) and avoids the model learning client-specific quirks instead of generalizable signal.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

df['label'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = ['search_volume', 'competition', 'cpc', 'word_count', 
                 'impressions_90d', 'clicks_90d', 'sessions_90d', 
                 'engaged_sessions_90d', 'scroll_events_90d',
                 'content_age_days', 'days_since_last_update',
                 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

df_clean = df.dropna(subset=feature_cols + ['label'])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_clean, groups=df_clean['client_id']))

train = df_clean.iloc[train_idx]
test = df_clean.iloc[test_idx]

print("Train:", train.shape, "Test:", test.shape)
print("Client overlap:", set(train['client_id']) & set(test['client_id']))

Train: (14234, 45) Test: (5663, 45)
Client overlap: set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training a Random Forest on the grouped split, then comparing Precision@50 against the ML-07 baseline on the same test set and metric.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

X_train, y_train = train[feature_cols], train['label']
X_test, y_test = test[feature_cols], test['label']

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

test = test.copy()
test['model_score'] = rf.predict_proba(X_test)[:, 1]

def precision_at_k(df_scored, score_col, k=50):
    top_k = df_scored.sort_values(score_col, ascending=False).head(k)
    return top_k['label'].mean()

model_p50 = precision_at_k(test, 'model_score', k=50)

# Baseline recreated on the SAME test set for a fair comparison
test['baseline_score'] = (np.log1p(test['impressions_90d']) * 0.7) + ((test['days_since_last_update'] / 365) * 0.3)
baseline_p50 = precision_at_k(test, 'baseline_score', k=50)

print(f"Baseline Precision@50: {baseline_p50:.3f}")
print(f"Random Forest Precision@50: {model_p50:.3f}")
print(f"AUC: {roc_auc_score(y_test, test['model_score']):.3f}")

Baseline Precision@50: 0.520
Random Forest Precision@50: 0.800
AUC: 0.622


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the model leans on:** permutation importance shows `content_age_days` (0.039) as by far the strongest signal, more than double the second-place feature, `impressions_90d` (0.020). This is a genuine surprise: the ML-07 baseline was built around `impressions_90d` and `days_since_last_update`, but the model found overall content age (not recency of the last edit) far more predictive.

**A confirmed weakness from ML-07 carries over here:** `days_since_last_update`, the signal the baseline weighted at 30%, has negative importance (-0.009) in the trained model, meaning shuffling it does not hurt performance, it may even help slightly. This matches the MIXED verdict from the ML-07 signal audit: staleness alone was never a reliable signal, and the model confirms it by essentially ignoring it in favor of `content_age_days` and `impressions_90d`.

**Error pattern:** most features cluster near zero or slightly negative importance, meaning the model's real signal comes from just two features (content age and impressions), the rest add noise more than predictive power. This suggests the next iteration should simplify around those two signals rather than feeding in all 15, extra features are not improving generalization here.

**Interpretation, not causation:** this shows content age and visibility are associated with decline in this sample, it does not establish that older content causes decline, older pages may simply belong to older topics or campaigns that naturally lost relevance for unrelated reasons.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

importance_df

,feature,importance
9,content_age_days,0.038690
4,impressions_90d,0.019760
12,avg_position,0.005015
14,scroll_rate,0.001766
3,word_count,0.000530
7,engaged_sessions_90d,0.000124
13,engagement_rate,-0.000530
11,ctr,-0.000954
8,scroll_events_90d,-0.001218
2,cpc,-0.001254


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.